In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
import joblib
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/kpneu_project")

MODEL_PATH = ROOT / "models/rf_scaffold_v1.joblib"
INPUT_PATH = ROOT / "results/day12_v2/day12_survivors.csv"
AD_PATH = ROOT / "results/day6_ad_threshold.json"

# Check files
for path in [MODEL_PATH, INPUT_PATH, AD_PATH]:
    print(path.relative_to(ROOT), "exists:", path.exists())

# Inspect the saved model
model_obj = joblib.load(MODEL_PATH)
print("\nModel object type:", type(model_obj))

if isinstance(model_obj, dict):
    print("Model dictionary keys:", list(model_obj.keys()))
    for key, value in model_obj.items():
        if key not in ["model", "estimator", "rf", "classifier"]:
            print(f"{key}: {str(value)[:300]}")
else:
    print("Model parameters:", model_obj.get_params())

# Inspect Day 12 input
df = pd.read_csv(INPUT_PATH)
print("\nInput shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3).to_string(index=False))

# Inspect the Day 6 threshold record
print("\nDay 6 AD threshold record:")
print(AD_PATH.read_text())

models/rf_scaffold_v1.joblib exists: True
results/day12_v2/day12_survivors.csv exists: True
results/day6_ad_threshold.json exists: True

Model object type: <class 'dict'>
Model dictionary keys: ['model', 'ad_threshold', 'cutoff_ug_mL', 'features', 'seed', 'train_split', 'sklearn']
ad_threshold: 0.3
cutoff_ug_mL: 16
features: Morgan r2 2048
seed: 42
train_split: scaffold
sklearn: 1.6.1

Input shape: (13203, 19)
Columns: ['smiles', 'maxsim_core_file', 'maxsim_all_file', 'pool', 'can', 'maxsim_all', 'maxsim_core', 'exact_train', 'MW', 'logP', 'TPSA', 'HBD', 'HBA', 'RotB', 'heavy_atoms', 'ro5_violations', 'pains_alerts', 'brenk_alerts', 'sa_score']
                               smiles  maxsim_core_file  maxsim_all_file pool                                   can  maxsim_all  maxsim_core  exact_train      MW   logP  TPSA  HBD  HBA  RotB  heavy_atoms  ro5_violations  pains_alerts  brenk_alerts  sa_score
    Brc1cc(Br)c2c(c1)CC(COc1cccnc1)C2          0.245902         0.245902 T1.0     Brc1cc(

In [3]:
!pip install -q rdkit
import sklearn
print("sklearn:", sklearn.__version__, "(must be 1.6.1)")

sklearn: 1.6.1 (must be 1.6.1)


In [4]:
import numpy as np, pandas as pd, json, joblib
from pathlib import Path
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import roc_auc_score
RDLogger.DisableLog('rdApp.*')

ROOT = Path("/content/drive/MyDrive/kpneu_project")
MODEL_PATH = ROOT / "models/rf_scaffold_v1.joblib"
INPUT_PATH = ROOT / "results/day12_v2/day12_survivors.csv"
AD_PATH    = ROOT / "results/day6_ad_threshold.json"
OUT13 = ROOT / "results/day13_v2"
OUT13.mkdir(parents=True, exist_ok=True)

model = joblib.load(MODEL_PATH)['model']
AD_THRESHOLD = json.loads(AD_PATH.read_text())['ad_threshold']
assert AD_THRESHOLD == 0.4, AD_THRESHOLD
LOW_CONF = 0.5     # extra annotation only
TOP_N = 10000      # plan: keep top 5-10k

# ---- sanity check: reproduce Day 5 test AUC ----
split = pd.read_csv(ROOT / "data/splits/split_v1.csv")
fp_saved = np.load(ROOT / "data/features/morgan_r2_2048_v1.npy")
print(split['split_scaffold'].value_counts())
lab = split['split_scaffold'].astype(str).str.lower()
te = (lab == 'test').values
auc = roc_auc_score(split.loc[te, 'active'].astype(int), model.predict_proba(fp_saved[te])[:, 1])
print(f"Re-scored test AUC: {auc:.3f} (Day 5 reported 0.867)\n")

# ---- score the Day 12 survivors ----
surv = pd.read_csv(INPUT_PATH)
for c in ['pains_alerts', 'brenk_alerts']:
    surv[c] = surv[c].fillna('')
print("survivors:", len(surv), "(expect 13,203)")

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
mols = [Chem.MolFromSmiles(s) for s in surv['can']]
print("parse failures:", sum(m is None for m in mols))
X = np.vstack([gen.GetFingerprintAsNumPy(m).astype(np.uint8) for m in mols])
surv['qsar_prob'] = model.predict_proba(X)[:, 1]
print(surv['qsar_prob'].describe().round(3))
print("share with prob >= 0.5:", round((surv.qsar_prob >= 0.5).mean() * 100, 1), "%")
print("share with prob >= 0.8:", round((surv.qsar_prob >= 0.8).mean() * 100, 1), "%")

split_scaffold
train    11366
test      1422
val       1420
Name: count, dtype: int64
Re-scored test AUC: 0.867 (Day 5 reported 0.867)

survivors: 13203 (expect 13,203)
parse failures: 0
count    13203.000
mean         0.614
std          0.274
min          0.006
25%          0.354
50%          0.639
75%          0.884
max          1.000
Name: qsar_prob, dtype: float64
share with prob >= 0.5: 60.2 %
share with prob >= 0.8: 37.0 %


In [5]:
tr = (lab == 'train').values
assert tr.sum() == 11366, tr.sum()

Xt = fp_saved[tr].astype(np.float32); ct = Xt.sum(1)
Xg = X.astype(np.float32);            cg = Xg.sum(1)
ad = np.zeros(len(Xg), dtype=np.float32)
for s in range(0, len(Xg), 1000):
    e = min(s + 1000, len(Xg))
    inter = Xg[s:e] @ Xt.T
    ad[s:e] = (inter / np.maximum(cg[s:e, None] + ct[None, :] - inter, 1)).max(1)
surv['ad_sim'] = ad
surv['outside_ad'] = surv['ad_sim'] < AD_THRESHOLD
surv['low_confidence'] = surv['ad_sim'] < LOW_CONF

# cross-check 20 random molecules with RDKit
from rdkit import DataStructs
train_fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in split.loc[tr, 'can_smiles']]
idx = np.random.default_rng(42).choice(len(surv), 20, replace=False)
diff = [abs(max(DataStructs.BulkTanimotoSimilarity(gen.GetFingerprint(mols[i]), train_fps)) - surv.loc[i, 'ad_sim']) for i in idx]
print("Max difference vs RDKit on 20 molecules:", round(max(diff), 5), "(should be ~0)\n")

print(f"Outside AD (<{AD_THRESHOLD}): {int(surv.outside_ad.sum()):,} ({surv.outside_ad.mean()*100:.1f}%)")
print(f"Low confidence (<{LOW_CONF}): {int(surv.low_confidence.sum()):,} ({surv.low_confidence.mean()*100:.1f}%)")
print("Day 6 reference on real val+test molecules: 10.8% outside AD\n")
print(surv['ad_sim'].describe().round(3))
print("\nMean QSAR prob inside AD :", round(surv.loc[~surv.outside_ad, 'qsar_prob'].mean(), 3))
print("Mean QSAR prob outside AD:", round(surv.loc[surv.outside_ad, 'qsar_prob'].mean(), 3))

from scipy.stats import spearmanr
print("Spearman QSAR prob vs maxsim_core:", np.round(spearmanr(surv['qsar_prob'], surv['maxsim_core']), 3))

Max difference vs RDKit on 20 molecules: 0.0 (should be ~0)

Outside AD (<0.4): 3,658 (27.7%)
Low confidence (<0.5): 5,464 (41.4%)
Day 6 reference on real val+test molecules: 10.8% outside AD

count    13203.000
mean         0.534
std          0.169
min          0.147
25%          0.380
50%          0.553
75%          0.683
max          1.000
Name: ad_sim, dtype: float64

Mean QSAR prob inside AD : 0.724
Mean QSAR prob outside AD: 0.328
Spearman QSAR prob vs maxsim_core: [0.811 0.   ]


In [6]:
surv = surv.sort_values('qsar_prob', ascending=False, kind='mergesort').reset_index(drop=True)
surv['qsar_rank'] = np.arange(1, len(surv) + 1)
surv.to_csv(OUT13 / 'day13_scored_all.csv', index=False)

for k in [5000, 10000]:
    sub = surv.head(k)
    print(f"top {k:>6,}: QSAR prob at last rank {sub['qsar_prob'].iloc[-1]:.3f} | "
          f"outside AD {int(sub.outside_ad.sum()):,} ({sub.outside_ad.mean()*100:.1f}%) | "
          f"low confidence {int(sub.low_confidence.sum()):,} ({sub.low_confidence.mean()*100:.1f}%)")

top = surv.head(TOP_N).copy()
print(f"\nKept {len(top):,} | QSAR prob range {top.qsar_prob.min():.3f} to {top.qsar_prob.max():.3f}")

top  5,000: QSAR prob at last rank 0.790 | outside AD 8 (0.2%) | low confidence 164 (3.3%)
top 10,000: QSAR prob at last rank 0.346 | outside AD 1,411 (14.1%) | low confidence 2,666 (26.7%)

Kept 10,000 | QSAR prob range 0.346 to 1.000


In [7]:
from rdkit.Chem.Scaffolds import MurckoScaffold

def summarise(d, name):
    return pd.Series({
        'n': len(d),
        'min_qsar_prob': d.qsar_prob.min(),
        'outside_AD_%': d.outside_ad.mean() * 100,
        'low_conf_%': d.low_confidence.mean() * 100,
        'maxsim_all<0.6_%': (d.maxsim_all < 0.6).mean() * 100,
        'maxsim_all<0.7_%': (d.maxsim_all < 0.7).mean() * 100,
        'median_maxsim_all': d.maxsim_all.median(),
        'median_SA': d.sa_score.median(),
        'median_MW': d.MW.median(),
        'median_RotB': d.RotB.median(),
        'Ro5_0_violations_%': (d.ro5_violations == 0).mean() * 100,
    }, name=name)

cmp = pd.concat([summarise(surv.head(5000), 'top5000'),
                 summarise(surv.head(10000), 'top10000'),
                 summarise(surv, 'all13203')], axis=1).round(2)
print(cmp)

def n_scaf(d):
    return len({MurckoScaffold.MurckoScaffoldSmiles(smiles=s) for s in d['can']})
for k in [5000, 10000, len(surv)]:
    print(f"top {k:>6,}: distinct Murcko scaffolds = {n_scaf(surv.head(k)):,}")

                    top5000  top10000  all13203
n                   5000.00  10000.00  13203.00
min_qsar_prob          0.79      0.35      0.01
outside_AD_%           0.16     14.11     27.71
low_conf_%             3.28     26.66     41.38
maxsim_all<0.6_%      19.64     47.53     59.52
maxsim_all<0.7_%      56.30     74.12     80.13
median_maxsim_all      0.69      0.61      0.55
median_SA              3.27      3.25      3.14
median_MW            405.45    409.86    397.54
median_RotB            4.00      4.00      4.00
Ro5_0_violations_%    83.68     76.67     77.26
top  5,000: distinct Murcko scaffolds = 2,237
top 10,000: distinct Murcko scaffolds = 6,091
top 13,203: distinct Murcko scaffolds = 8,752


In [8]:
import time
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors

DO_3D = True   # False skips the slow 3D globularity proxy

amine = Chem.MolFromSmarts('[NX3;H2;!$(N-[#6]=[O,S,N]);!$(N-S);!$(N-a);!$(N-[#7,#8])][CX4]')
top = surv.head(TOP_N).copy()
tmols = [Chem.MolFromSmiles(s) for s in top['can']]

top['has_primary_amine'] = [int(m.HasSubstructMatch(amine)) for m in tmols]
top['rigid_rotb_le5'] = (top['RotB'] <= 5).astype(int)
top['frac_rotb'] = (top['RotB'] / top['heavy_atoms']).round(3)

def spherocity(m):
    mh = Chem.AddHs(m)
    ps = AllChem.ETKDGv3(); ps.randomSeed = 42
    if AllChem.EmbedMolecule(mh, ps) != 0:
        return np.nan
    return rdMolDescriptors.CalcSpherocityIndex(mh)

if DO_3D:
    t0, vals = time.time(), []
    for i, m in enumerate(tmols, 1):
        vals.append(spherocity(m))
        if i % 1000 == 0:
            print(f"  {i:,} / {len(tmols):,} | {time.time()-t0:.0f}s")
    top['globularity_proxy'] = np.round(vals, 3)
    print("3D failures:", int(top['globularity_proxy'].isna().sum()))

print("Primary amine:", int(top.has_primary_amine.sum()), "| RotB <= 5:", int(top.rigid_rotb_le5.sum()))
top.to_csv(OUT13 / f'day13_top{TOP_N}.csv', index=False)

f12 = pd.read_csv(ROOT / 'results/day12_v2/day12_funnel.csv')[['step', 'n_remaining']]
extra = pd.DataFrame([
    {'step': 'Day 13: scored with QSAR (all Day 12 survivors)', 'n_remaining': len(surv)},
    {'step': f'Day 13: kept top {TOP_N:,} by QSAR probability', 'n_remaining': len(top)},
])
ft = pd.concat([f12, extra], ignore_index=True)
ft['removed_at_step'] = (-ft['n_remaining'].diff()).fillna(0).astype(int)
ft.to_csv(OUT13 / 'funnel_through_day13.csv', index=False)
print(ft.to_string(index=False))

  1,000 / 10,000 | 65s
  2,000 / 10,000 | 148s
  3,000 / 10,000 | 221s
  4,000 / 10,000 | 406s
  5,000 / 10,000 | 671s
  6,000 / 10,000 | 1042s
  7,000 / 10,000 | 1410s
  8,000 / 10,000 | 1737s
  9,000 / 10,000 | 2238s
  10,000 / 10,000 | 2629s
3D failures: 90
Primary amine: 1955 | RotB <= 5: 6923
                                           step  n_remaining  removed_at_step
     Start: valid unique molecules from file(s)        39201                0
           Valid and canonical (RDKit re-parse)        39201                0
                Deduplicated (canonical SMILES)        39201                0
   Removed exact matches to known train actives        37398             1803
  Removed near-duplicates (max Tanimoto >= 0.8)        33439             3959
                    Property window: MW 150-700        31711             1728
         Property window: rotatable bonds <= 12        30055             1656
                                  Removed PAINS        28293             1762

In [9]:
# 1) fresh vs saved fingerprints, first 300 rows
fresh = np.vstack([gen.GetFingerprintAsNumPy(Chem.MolFromSmiles(s)).astype(np.uint8)
                   for s in split['can_smiles'].iloc[:300]])
print("Fresh fingerprints match saved .npy (300 rows):",
      np.array_equal(fresh, fp_saved[:300].astype(np.uint8)))

# 2) ties at the top-N cut
p = surv['qsar_prob'].values
cut = p[TOP_N - 1]
tied = np.isclose(p, cut)
print(f"QSAR prob at rank {TOP_N:,}: {cut:.3f} | molecules with this same score: {int(tied.sum())} "
      f"| kept: {int(tied[:TOP_N].sum())}")

# 3) threshold actually used
print("AD_THRESHOLD used:", AD_THRESHOLD, "| outside AD in all 13,203:", int(surv.outside_ad.sum()))

Fresh fingerprints match saved .npy (300 rows): True
QSAR prob at rank 10,000: 0.346 | molecules with this same score: 1 | kept: 1
AD_THRESHOLD used: 0.4 | outside AD in all 13,203: 3658


In [10]:
import numpy as np, pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

# 1) Fingerprints (same settings as before)
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
X = np.array([gen.GetFingerprintAsNumPy(Chem.MolFromSmiles(s)) for s in top['can']], dtype=np.float32)

# 2) Tanimoto distance matrix with numpy (about 1 GB for 10,000 molecules)
inter = X @ X.T
n = X.sum(1)
D = 1.0 - inter / (n[:, None] + n[None, :] - inter)
del inter

# 3) Butina clustering written with numpy
def butina(D, cutoff):
    N = D.shape[0]
    nb = D <= cutoff                     # neighbour table (includes the molecule itself)
    counts = nb.sum(1).astype(np.int64)
    free = np.ones(N, bool)
    labels = np.full(N, -1)
    centroid = np.zeros(N, bool)
    cid = 0
    while free.any():
        c = np.where(free, counts, -1).argmax()
        members = np.where(nb[c] & free)[0]
        labels[members] = cid
        centroid[c] = True
        free[members] = False
        counts -= nb[members].sum(0)
        cid += 1
    return labels, centroid

# 4) Run 3 cutoffs, save assignments, record numbers
rows = []
for cut in [0.5, 0.6, 0.7]:
    labels, centroid = butina(D, cut)
    size = pd.Series(labels).map(pd.Series(labels).value_counts()).values
    out = pd.DataFrame({'row_in_top': np.arange(len(top)), 'can': top['can'].values,
                        'cluster_id': labels, 'cluster_size': size, 'is_centroid': centroid.astype(int)})
    out.to_csv(OUT13 / f'day13_clusters_cut{cut}.csv', index=False)
    vc = pd.Series(labels).value_counts()
    rows.append({'cutoff': cut, 'n_clusters': len(vc), 'singletons': int((vc == 1).sum()), 'largest_cluster': int(vc.max())})
    print(rows[-1])

summary = pd.DataFrame(rows)
summary.to_csv(OUT13 / 'day13_cluster_summary.csv', index=False)

# 5) Funnel rows (these are cluster counts, not molecules removed)
ft = pd.read_csv(OUT13 / 'funnel_through_day13.csv')
extra = pd.DataFrame([{'step': f'Day 13: Butina clusters at cutoff {r.cutoff} (clusters, not removals)',
                       'n_remaining': r.n_clusters, 'removed_at_step': np.nan} for r in summary.itertuples()])
pd.concat([ft, extra], ignore_index=True).to_csv(OUT13 / 'funnel_through_day13_with_clusters.csv', index=False)
print(summary.to_string(index=False))

{'cutoff': 0.5, 'n_clusters': 3488, 'singletons': 2978, 'largest_cluster': 1067}
{'cutoff': 0.6, 'n_clusters': 2087, 'singletons': 1716, 'largest_cluster': 2883}
{'cutoff': 0.7, 'n_clusters': 979, 'singletons': 726, 'largest_cluster': 4691}
 cutoff  n_clusters  singletons  largest_cluster
    0.5        3488        2978             1067
    0.6        2087        1716             2883
    0.7         979         726             4691


In [11]:
for cut in [0.5, 0.6, 0.7]:
    c = pd.read_csv(OUT13 / f'day13_clusters_cut{cut}.csv')
    sizes = c.groupby('cluster_id').size()
    ok_sizes = (c['cluster_size'] == c['cluster_id'].map(sizes)).all()
    one_centroid = (c.groupby('cluster_id')['is_centroid'].sum() == 1).all()
    print(cut, "| rows:", len(c), "| clusters:", c['cluster_id'].nunique(),
          "| sizes consistent:", ok_sizes, "| one centroid per cluster:", one_centroid)

0.5 | rows: 10000 | clusters: 3488 | sizes consistent: True | one centroid per cluster: True
0.6 | rows: 10000 | clusters: 2087 | sizes consistent: True | one centroid per cluster: True
0.7 | rows: 10000 | clusters: 979 | sizes consistent: True | one centroid per cluster: True
